In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles"

Mounted at /content/drive


In [ ]:
import os
import random
import shutil
import rasterio

In [ ]:
#### Step 1 - Randomly Sample 100 tiles from Tile Bank of Cambridge Town Plans 1Ed ####

SEED = 123
random.seed(SEED)

from pathlib import Path
Tile_Bank = Path(base_dir) / "TownPlan_1Ed_1_500" / "camb" / "Cambridge"
tif_files = list(Tile_Bank.glob('*.tif'))
print(tif_files)

Samples_to_annotate = "/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate"
os.makedirs(Samples_to_annotate, exist_ok= True) # make the new directory for geotiffs to sample

#Samples = random.sample(tif_files, 100) # sample 100 geotiffs to annotate

for s in Samples:  # Samples = list of .tif files
   stem = Path(s).stem # 100 sample geotiffs without suffix to copy .tif .tfw and .tab files over
   for file in Tile_Bank.glob(stem + ".*"):
        shutil.copy(file, Samples_to_annotate) # copy files to samples to annotate folder

[PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl440580-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl440599-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl443596-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl447597-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl446597-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl443582-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439596-1.tif'), PosixPath('/content/drive/MyDrive/Colab_Data/Town_Plan

#### Step 2 - Annotate the 100 Sample Tiles in QGIS ####

Classes
*   0 - L.P
*   1 - L.
*   2 - Post or Posts (pprobably not lamp posts but boulders)
*   3 - partial L.P (cut of tile edge)

To create annotations.gpkg:
: Click the

1.   Layer > Create Layer > New GeoPackage Layer - then find where you want to save the .gpkg project
2.   Set a Table Name (e.g. lamppost_annotations)
3.   Set Type to Polygon (n.b. this is important because YOLO training requires bounding boxes, rectangles - we can convert random shaped polygons to rectangles easily later)
4.   Set CRS - e.g. EPSG:27700 - OSGB36 / British National Grid
5.   Add New Field class_id and select 32 bit integer
6.   Click the Add to Fields List button (class_id should be in the list below the "id" field.) - then press Okay.

To set default value:
1.   Open Properties of annotations.gpkg layer (right click in the Layers Panel)
2.   Click into the Attributes Form Tab
3.   Click on class_id under the "Fields" list
4.   Set "Widget Type" to Text Edit
5.   In the Defaults section, enter the "Default value" as 0.
6.   (If you have done some already and you want to set all of those to 0, then check the "Apply default value on update"
7.   Press Apply to activate the changes

To check CRS is all good:
1.   Drag one .tif into QGIS.
2.   Check the CRS shown in the CRS display (bottom right of window)
3.   Hover your mouse over the map and check coordinate display numbers look right (e.g. Eastings/Northings)

Annotation Process
1) Create the annotations.gpkg with a Polygon geometry and a 32-bit Integer field named class_id .
2) Load your maps as rasters
3) Toggle Editing on the annotations.gpkg later and activate it
4) Draw a box around an "LP". When the box pops up, type the correct class into the class_id field (e.g. 0 for L.P)
5) Once you've added some polygons, remember to press Save Edits.
(n.b. you can only digitise/add polygons when the anno

In [ ]:
# Step 3 - Split Sample tiles into smaller tiles
# This script will chop each of the sample tiles (3000x3000) into overlapping smaller tiles within (1024x1024)

from pathlib import Path
from rasterio.windows import Window

Big_tiles = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate")
Smaller_Tiles = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_as_smaller_tiles")
os.makedirs(Smaller_Tiles, exist_ok=True) # make the new directory for smaller tiles of geotiffs

# Smaller tile settings
small_tiles_size = 1024  # small tile size in pixels
tile_overlap = 100     # pixels that overlap between smaller tiles in the big tile
step = small_tiles_size - tile_overlap # move 924 pixels and then cut tile

for geotiff in Big_tiles.glob("*.tif"):
  with rasterio.open(geotiff) as tile: # read in each geotiff file

    for x in range(0, tile.width, step): # x cutting point
      for y in range(0, tile.height, step): # y cutting point
        window = Window(x, y, small_tiles_size, small_tiles_size) # specify the small tile window
        small_tile = tile.read(window=window)# read in the part of the tile that is in the small tile window
        transform = tile.window_transform(window) # fix the spatial coordinates because when reading in using a window, the top left corner changes

        small_tile_filename = Smaller_Tiles / f"{geotiff.stem}_{x}_{y}.tif"

        with rasterio.open(small_tile_filename,
                         'w', # "w" = write new raster
                         driver="GTiff", # "GTiff"  = geotiff file type
                         height= small_tile.shape[1], # height =
                         width= small_tile.shape[2],
                         count=tile.count, # get the same number of bands as the original geotiff
                         dtype=small_tile.dtype, # set the pixel data type
                         crs=tile.crs, # keep crs as original
                         transform=transform, # get correct spatial coordinates
             ) as dst: # save the tile
                dst.write(small_tile)


In [ ]:
# Step 4 - Convert Annotations into YOLO Labels

import geopandas
import os
import rasterio
from shapely.geometry import box
from pathlib import Path

annotations_gpkg = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_annotations.gpkg")
Smaller_Tiles = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_as_smaller_tiles")
Labels = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_labels")
os.makedirs(Labels, exist_ok=True) # make the new directory for YOLO labels

annotations = geopandas.read_file(annotations_gpkg) # read in annotations gpkg
small_tiles_size = 1024  # small tile size in pixels

for smalltiffpath in Smaller_Tiles.glob("*.tif"): # for each small tile
# smalltiffpath = list(Smaller_Tiles.glob("*.tif"))

# use camb-tl441595-1_0_1848.tif as a test - there is one lamppost there
# smalltiffpath = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_as_smaller_tiles/camb-tl441595-1_0_1848.tif")

  with rasterio.open(smalltiffpath) as smalltiff: # read in each small geotiff file
      #print(smalltiffpath)
      bounds_geom = box(*smalltiff.bounds) # get the bounds of smalltiff
      #print(bounds_geom)
      intersecting = annotations[annotations.intersects(bounds_geom)] # identify any annotations that fall within bounds
      #print(intersecting)
      label_file = Labels / f"{smalltiffpath.stem}.txt"
      label_file.touch(exist_ok=True)
      #print(label_file)
      with open(label_file, "w") as f:
          if len(intersecting) > 0: # if there is an annotation
                  for idx, row in intersecting.iterrows(): # Use iterrows to get the class_id
                    class_id = row['class_id']

                    if class_id == 2: # Skip Class 2 (Boulder posts) entirely
                        continue

                    # Get the intersection (in case it hangs off the edge)
                    clipped_geom = row.geometry.intersection(bounds_geom) # restrict bounding box to just tile
                    minx, miny, maxx, maxy = clipped_geom.bounds # get bounding box
                    # Map coordinates to pixel numbers
                    r1, c1 = smalltiff.index(minx, maxy)
                    r2, c2 = smalltiff.index(maxx, miny)
                    # Convert pixels to YOLO normalized decimals (0.0 to 1.0)
                    # Get centrepoint of annotation box
                    x_center = ((c1 + c2) / 2) / small_tiles_size
                    y_center = ((r1 + r2) / 2) / small_tiles_size
                    # Get width and height of annotation box
                    w = abs(c2 - c1) / small_tiles_size
                    h = abs(r2 - r1) / small_tiles_size
                    # Write: [class] [x_center] [y_center] [width] [height]
                    f.write(f"{class_id} {x_center} {y_center} {w} {h}\n")

In [ ]:
# Step 5 - Set up Training and Validation

from pathlib import Path

annotations_gpkg = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_annotations.gpkg")
Smaller_Tiles = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_as_smaller_tiles")
Labels = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_labels")
YOLO_dataset = Path("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset")

train_ratio = 0.8

os.makedirs(f"{YOLO_dataset}/images/train", exist_ok=True)
os.makedirs(f"{YOLO_dataset}/images/val", exist_ok=True)
os.makedirs(f"{YOLO_dataset}/labels/train", exist_ok=True)
os.makedirs(f"{YOLO_dataset}/labels/val", exist_ok=True)

smaller_tiles = list(Smaller_Tiles.glob("*.tif"))

# All small tiles from the original big tile need to be together in either TRAINING or VALIDATION
big_tile_id = {}

for smaller_tile in smaller_tiles: # group smaller tiles by original big tile
    prefix = os.path.basename(smaller_tile).split("_")[0] # get the big tiles for each small tile
    big_tile_id.setdefault(prefix, []).append(smaller_tile)
    # if the big tile prefix doesn't exist in the big_tile_id list yet,
    # add it to the list and then append the name of the small tile
    # such that for each big tile (KEY), there is a list of associated small tiles (VALUES)

# print(big_tile_id)

keys = list(big_tile_id.keys())
# print(keys)
random.shuffle(keys)
# print(keys) shuffled KEYS

split_index = int(len(keys) * train_ratio)
# print(split_index)
train_keys = keys[:split_index] # 0:79 is training
# print(train_keys)
val_keys = keys[split_index:] # 80:99 is validation
# print(val_keys)

small_tiles_training = []
for key in train_keys:
    small_tiles_training.extend(big_tile_id[key])
# print(small_tiles_training)
# len(small_tiles_training)

small_tiles_validation = []
for key in val_keys:
    small_tiles_validation.extend(big_tile_id[key])
# print(small_tiles_validation)
# len(small_tiles_validation)

for tile in smaller_tiles:
  label_file = Labels / f"{tile.stem}.txt" # give it the Path to label

  if tile in small_tiles_training: # if part of training dataset
    shutil.copy(tile, f"{YOLO_dataset}/images/train") # copy image
    shutil.copy(label_file, f"{YOLO_dataset}/labels/train") # copy labels
  elif tile in small_tiles_validation:
    shutil.copy(tile, f"{YOLO_dataset}/images/val") # copy image
    shutil.copy(label_file, f"{YOLO_dataset}/labels/val") # copy labels

print("Train/Val split complete.")

Train/Val split complete.


In [ ]:
# Step 6 - Set up YAML for YOLO Model training

import yaml

data = {
    'path': '/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'L_P',
        1: 'L_dot',
        2: 'Unused_Post', # The model will never see labels for this
        3: 'Partial_LP'
    },
    'channels': 1
}

with open("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/training_config.yaml", "w") as file:
    yaml.dump(data, file)

print("Data has been written to main folder")

Data has been written to main folder


In [ ]:
!nvidia-smi

Sat May 30 12:36:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.6 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.cuda.is_available())

from ultralytics import YOLO

True
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Step 7 - YOLO Model training

from ultralytics import YOLO

model = YOLO("yolo11m.pt")

model.train(
    data= "/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/training_config.yaml",
    epochs=80,
    imgsz=1024,
    batch=4,
    device=0,
    project="/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/runs",
    name="yolo11m_2026_05_15"
)

#import os
#import shutil

#src = "/content/runs/detect/train-2"
#dst = "/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/runs"

#os.makedirs(os.path.dirname(dst), exist_ok=True)

#shutil.copytree(src, dst, dirs_exist_ok=True)

print("Saved successfully")

Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/training_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_2026_05_15-2, nbs=64, nms=False, opset=None, o

In [ ]:
# Step 8

from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/runs/yolo11m_2026_05_15-2/weights/best.pt")

results = model.predict(
    source="/content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/",
    imgsz=1024,
    conf=0.01,
    batch=16,
    device=0,
    save = True,
    save_txt = True,
    project="/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/predictions",
    name="yolo11m_2026_05_15-2"
)




image 1/1028 /content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439573-1.tif: 1024x1024 (no detections), 154.9ms
image 2/1028 /content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439574-1.tif: 1024x1024 (no detections), 154.9ms
image 3/1028 /content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439575-1.tif: 1024x1024 (no detections), 154.9ms
image 4/1028 /content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439576-1.tif: 1024x1024 (no detections), 154.9ms
image 5/1028 /content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439577-1.tif: 1024x1024 (no detections), 154.9ms
image 6/1028 /content/drive/MyDrive/Colab_Data/Town_Plans_1_500_Cambridge_Tiles/TownPlan_1Ed_1_500/camb/Cambridge/camb-tl439578-1.tif: 1024x1024 (no 

Model: YOLO11s
Epochs: 80
Image size: 1024
Channels: 1
Confidence used: 0.01 and 0.25
Main issue: false positives on letters/text

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/runs/weights/best.pt")

results = model.predict(
    source="/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/",
    imgsz=1024,
    conf=0.01,
    batch=16,
    device=0,
    save = True,
    save_txt = True,
    project="/content/drive/MyDrive/Colab_Data/TownPlan_1Ed_YOLO_dataset/predictions",
    name="predictions_on_TownPlan_1Ed_geotiffs_to_annotate"
)




image 1/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl439575-1.tif: 1024x1024 (no detections), 99.8ms
image 2/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl439576-1.tif: 1024x1024 (no detections), 99.8ms
image 3/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl440576-1.tif: 1024x1024 (no detections), 99.8ms
image 4/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl440590-1.tif: 1024x1024 (no detections), 99.8ms
image 5/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl441576-1.tif: 1024x1024 (no detections), 99.8ms
image 6/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl441577-1.tif: 1024x1024 (no detections), 99.8ms
image 7/100 /content/drive/MyDrive/Colab_Data/TownPlan_1Ed_geotiffs_to_annotate/camb-tl441582-1.tif: 1024x1024 (no detections), 99.8ms
image 8/100 /content/drive/MyDrive/Colab_Data/TownPlan